In [8]:
import torch

In [15]:
import numpy as np
import pandas as pd
import wfdb
import os
import sys
from scipy import signal
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt
import pywt

In [ ]:
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a
 
def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data)
    return y

def calc_baseline(signal):
    """
    Calculate the baseline of signal.

    Args:
        signal (numpy 1d array): signal whose baseline should be calculated


    Returns:
        baseline (numpy 1d array with same size as signal): baseline of the signal
    """
    ssds = np.zeros((3))

    cur_lp = np.copy(signal)
    iterations = 0
    while True:
        # Decompose 1 level
        lp, hp = pywt.dwt(cur_lp, "db4")

        # Shift and calculate the energy of detail/high pass coefficient
        ssds = np.concatenate(([np.sum(hp ** 2)], ssds[:-1]))

        # Check if we are in the local minimum of energy function of high-pass signal
        if ssds[2] > ssds[1] and ssds[1] < ssds[0]:
            break

        cur_lp = lp[:]
        iterations += 1

    # Reconstruct the baseline from this level low pass signal up to the original length
    baseline = cur_lp[:]
    for _ in range(iterations):
        baseline = pywt.idwt(baseline, np.zeros((len(baseline))), "db4")

    return baseline[: len(signal)]

# Reshape a numpy array 'a' of shape (x) to form shape((n - window_size) // step + 1, window_size))
def rolling_window(a, window, step):
    shape = a.shape[:-1] + ((a.shape[-1] - window + 1)//step, window)
    strides = (a.strides[0] * step,) + (a.strides[-1],)
    return np.lib.stride_tricks.as_strided(a, shape=shape, strides=strides)

def preprocessing(ecg):
    ecg = butter_bandpass_filter(ecg, 0.5, 30, 360, 5)
    baseline = calc_baseline(ecg)
    ecg -= baseline
    return ecg

In [ ]:
def get_data(db_dir='/home/server2/Desktop/data_training_afib/INPUT/mitdb'):
    records = []
    for file in os.listdir(db_dir):
        if file.endswith('.hea'):
            record_name = os.path.splitext(file)[0]
            record_path = os.path.join(db_dir, record_name)

            record = wfdb.rdrecord(record_path)
            ann = wfdb.rdann(record_path, extension='atr')

            original_signal = record.p_signal.T[1]

            freq_sampling = record.fs
            target_fs = 256

            # Calculate the resampling factor
            resampling_factor = target_fs / freq_sampling

            # Resample the signal
            signals = signal.resample(original_signal, int(len(original_signal) * resampling_factor))
            
            # Filter the signal
            filtered_signal = preprocessing(signals)

            sample, symbol, aux_note = ann.sample * target_fs // freq_sampling, np.array(ann.symbol), np.array(ann.aux_note)
            
            WINDOW_SIZE = 30 #s
            STRIDE = target_fs * 40 #sample
            x = rolling_window(filtered_signal, target_fs * WINDOW_SIZE, STRIDE)
            selected_samples = []
            y = []
            has_afib = 0
            for (i, val) in enumerate(x):
                fl = 0
                random_num = np.random.rand()

                if(random_num >= 1):
                    fl = 1
                
                if fl == 1:
                    continue
                
                selected_samples.append(i)

                start_idx = STRIDE * i
                oneSecSegment = rolling_window(val, target_fs, target_fs - 7)
                # print(len(oneSecSegment))
                y_seg = []
                for (j, seg) in enumerate(oneSecSegment):
                    start_idx_seg = start_idx + j * target_fs
                    end_idx_seg = start_idx_seg + target_fs
                    ref_idx = np.array(np.where((sample >= start_idx_seg) & (sample <= end_idx_seg))[0])
                    ref_symbol = aux_note[ref_idx]
                    afib_sum = (ref_symbol == '(AFIB').sum()
                    other_sum = ((ref_symbol != '(AFIB') & (ref_symbol != '')).sum()
                
                    if afib_sum > 0:
                        has_afib = 1
                    elif other_sum > 0 and has_afib:
                        has_afib = 0
                    
                    if(has_afib):
                        y_seg.append(1)
                    else:
                        y_seg.append(0)
                y.append(y_seg)
                
            records.append((x[selected_samples][:], y))
    return records
records = get_data()

In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", device)

Device:  cuda
